### CELL 1: [POIN a.i] AKUISISI DATA — UNDUH PDF MENTAH
Scraper mengunduh file PDF dari web Mahkamah Agung dan menyimpannya ke `data/pdf/<nama>/` **tanpa ekstraksi teks**

In [ ]:
import os
import base64
import time
import random
import re
from bs4 import BeautifulSoup
from seleniumbase import Driver
from tqdm import tqdm  

# 1. Tentukan folder spesifik agar tidak bercampur
PDF_DATA_DIR = os.path.join("data", "pdf")
os.makedirs(PDF_DATA_DIR, exist_ok=True)

BASE_URL = "https://putusan3.mahkamahagung.go.id"

DIREKTORI_ENDPOINTS = [
    "/direktori/index/pengadilan/mahkamah-agung/kategori/peradilan-anak-abh-1.html"
]

# --- KONFIGURASI KONSISTENSI CBR ---
TAHUN_MINIMAL = 2018  # Batas bawah tahun putusan
TAHUN_MAKSIMAL = 2026 # Batas atas tahun putusan

CHECKPOINT_FILE = "progress_checkpoint_cbr.txt"
HISTORY_FILE = "riwayat_terunduh_cbr.txt"

URL_TERPROSES = set()
NOMOR_TERUNDUH = set()

if os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                NOMOR_TERUNDUH.add(line.strip())

JS_FETCH_PDF_ASYNC = """
var url      = arguments[0];
var callback = arguments[1];
var xhr = new XMLHttpRequest();
xhr.open('GET', url, true);
xhr.responseType = 'arraybuffer';
xhr.setRequestHeader('Accept', 'application/pdf,*/*');
xhr.onload = function() {
    if (xhr.status !== 200) { callback(null); return; }
    var bytes  = new Uint8Array(xhr.response);
    var binary = '';
    var chunk  = 8192;
    for (var i = 0; i < bytes.byteLength; i += chunk)
        binary += String.fromCharCode.apply(null, bytes.subarray(i, i + chunk));
    callback(window.btoa(binary));
};
xhr.onerror   = function() { callback(null); };
xhr.ontimeout = function() { callback(null); };
xhr.timeout   = 25000;
xhr.send();
"""

def ambil_metadata_putusan(soup):
    meta = {}
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) >= 2:
            kunci = tds[0].get_text(strip=True).lower().rstrip(":")
            nilai = tds[1].get_text(" ", strip=True)
            if kunci and nilai:
                meta[kunci] = nilai
    for dt in soup.find_all("dt"):
        kunci = dt.get_text(strip=True).lower().rstrip(":")
        dd = dt.find_next_sibling("dd")
        if dd:
            meta[kunci] = dd.get_text(" ", strip=True)
    return meta


def verifikasi_konsistensi_cbr(meta, teks_indeks=""):
    """
    Fungsi Filter Ketat: Hanya meloloskan KASASI murni dan masuk dalam rentang tahun target.
    """
    semua_teks = (" ".join(meta.values()) + " " + teks_indeks).upper()
    
    nomor = (
        meta.get("nomor", "") + 
        meta.get("no", "") + 
        meta.get("no putusan", "")
    ).upper()

    # 1. FILTER TINGKATAN: Harus Kasasi Murni (Mengandung huruf 'K' tapi BUKAN 'PK')
    # Pola regex memastikan ada kode 'K' yang berdiri sendiri di antara garis miring khas nomor MA
    adalah_kasasi = bool(re.search(r"/\s*K\s*/", nomor)) or ("KASASI" in semua_teks)
    adalah_pk = bool(re.search(r"/\s*PK\s*/", nomor)) or ("PENINJAUAN KEMBALI" in semua_teks)
    
    if adalah_pk or not adalah_kasasi:
        return False

    # Ekstraksi Tahun dari metadata untuk filter rentang waktu
    teks_tanggal = (meta.get("tanggal putus", "") + " " + meta.get("tahun", "") + " " + meta.get("tanggal musyawarah", ""))
    daftar_tahun = [int(t) for t in re.findall(r"\b(20\d{2})\b", teks_tanggal)]
    
    if daftar_tahun:
        tahun_putusan = max(daftar_tahun) # Ambil tahun paling valid
        # 2. FILTER TAHUN: Harus masuk dalam rentang yang ditentukan
        if not (TAHUN_MINIMAL <= tahun_putusan <= TAHUN_MAKSIMAL):
            print(f"[SKIP] Tahun putusan ({tahun_putusan}) di luar rentang {TAHUN_MINIMAL}-{TAHUN_MAKSIMAL}.")
            return False
    else:
        # Jika tahun tidak terdeteksi di teks metadata, kita coba intip dari nomor perkara (ex: 123 K/Pid.Sus/2020)
        tahun_dari_nomor = re.findall(r"\b(20\d{2})\b", nomor)
        if tahun_dari_nomor:
            tahun_putusan = int(tahun_dari_nomor[0])
            if not (TAHUN_MINIMAL <= tahun_putusan <= TAHUN_MAKSIMAL):
                return False
        else:
            # Jika sama sekali tidak ada data tahun, skip demi keamanan konsistensi data CBR
            print("[SKIP] Tidak menemukan data tahun putusan yang valid.")
            return False

    # 3. PROTEKSI PERADILAN BAWAH
    pola_bawah = r"\b(PENGADILAN NEGERI|PENGADILAN TINGGI|PENGADILAN AGAMA|PN|PT|PA)\b"
    ada_pengadilan_bawah = bool(re.search(pola_bawah, semua_teks)) or "/PN/" in semua_teks or "/PT/" in semua_teks

    return not ada_pengadilan_bawah


def ekstrak_link_khusus_ma(soup):
    links_valid = []
    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"]
        if "/putusan/" in href and "/download/" not in href and "/berkas/" not in href:
            
            parent_box = a_tag.find_parent("div") or a_tag.find_parent("td") or a_tag.find_parent("li")
            teks_konteks = parent_box.get_text(" ", strip=True).upper() if parent_box else ""
            
            if any(k in teks_konteks for k in ["PENGADILAN NEGERI", "PENGADILAN AGAMA"]):
                continue
            if re.search(r"\b(PN|PA)\b", teks_konteks):
                continue
                
            full_url = href if href.startswith("http") else f"{BASE_URL}{href}"
            
            if full_url in URL_TERPROSES:
                continue
                
            if (full_url, teks_konteks) not in links_valid:
                links_valid.append((full_url, teks_konteks))
                
    return links_valid


def ambil_links_dari_direktori(driver, base_path, page):
    if page == 1:
        url = f"{BASE_URL}{base_path}"
    else:
        cleaned_path = base_path.replace(".html", "")
        url = f"{BASE_URL}{cleaned_path}/page/{page}.html"
        
    print(f"\n[PAGE {page}] {url}")
    driver.uc_open_with_reconnect(url, reconnect_time=6)
    time.sleep(random.uniform(3.5, 4.5))

    soup = BeautifulSoup(driver.page_source, "html.parser")
    links_data = ekstrak_link_khusus_ma(soup)

    return links_data


def fetch_pdf_bytes(driver, url_pdf):
    try:
        driver.set_script_timeout(25)
        b64_result = driver.execute_async_script(JS_FETCH_PDF_ASYNC, url_pdf)
        if not b64_result:
            return None
        pdf_bytes = base64.b64decode(b64_result)
        if pdf_bytes[:4] != b"%PDF":
            return None
        return pdf_bytes
    except Exception:
        return None


def cari_url_pdf(soup_detail):
    kandidat_pdf  = []
    kandidat_lain = []
    for a in soup_detail.find_all("a", href=True):
        href = a["href"]
        teks = a.get_text(strip=True).lower()
        if any(k in href.lower() for k in ["download_file", "/pdf/", ".pdf"]) or \
           any(k in teks for k in ["pdf", "download", "unduh"]):
            url = href if href.startswith("http") else f"{BASE_URL}{href}"
            if "/pdf/" in url or ".pdf" in url:
                if url not in kandidat_pdf:
                    kandidat_pdf.append(url)
            else:
                if url not in kandidat_lain:
                    kandidat_lain.append(url)
    return kandidat_pdf + kandidat_lain


def jalankan_scraper_pdf(limit_data=40):
    global NOMOR_TERUNDUH, URL_TERPROSES
    
    downloaded_count        = len(NOMOR_TERUNDUH)
    skipped_count           = 0
    bukan_target_count      = 0
    endpoint_idx            = 0
    halaman_kosong_berturut = 0

    page = 1
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            try: page = int(f.read().strip())
            except ValueError: page = 1

    print("[START] Mengaktifkan Browser Scraper Konsisten...")
    driver = Driver(uc=True, headless=False, incognito=True)
    
    pbar = tqdm(total=limit_data, initial=min(downloaded_count, limit_data), desc="Progres Standarisasi Data CBR")

    try:
        while downloaded_count < limit_data:
            if halaman_kosong_berturut >= 3:
                endpoint_idx += 1
                if endpoint_idx >= len(DIREKTORI_ENDPOINTS):
                    print("[WARN] Proses selesai. Indeks direktori telah habis dipindai.")
                    break
                print(f"[SWITCH] Pindah ke endpoint: {DIREKTORI_ENDPOINTS[endpoint_idx]}")
                page                    = 1
                halaman_kosong_berturut = 0

            with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
                f.write(str(page))

            base_path = DIREKTORI_ENDPOINTS[endpoint_idx]
            links_putusan = ambil_links_dari_direktori(driver, base_path, page)

            print(f"[INFO] Terdeteksi {len(links_putusan)} kandidat pada halaman ini.")

            if not links_putusan:
                halaman_kosong_berturut += 1
                page += 1
                continue

            halaman_kosong_berturut = 0

            for url_detail, teks_indeks in links_putusan:
                if downloaded_count >= limit_data:
                    break

                URL_TERPROSES.add(url_detail)

                print(f"[->] Membuka Dokumen: {url_detail}")
                driver.uc_open_with_reconnect(url_detail, reconnect_time=5)
                time.sleep(random.uniform(3.0, 4.0))

                soup_detail = BeautifulSoup(driver.page_source, "html.parser")
                meta        = ambil_metadata_putusan(soup_detail)
                
                # MENGGUNAKAN FILTER BARU YANG KETAT UNTUK CBR
                if not verifikasi_konsistensi_cbr(meta, teks_indeks):
                    print("[SKIP] Gagal verifikasi ketat (Bukan Kasasi murni / Tahun tidak sesuai rentang).")
                    bukan_target_count += 1
                    continue

                nomor_perkara = meta.get("nomor") or meta.get("no") or meta.get("no putusan") or ""
                nomor_perkara_clean = re.sub(r"\s+", "", nomor_perkara).upper()
                
                if not nomor_perkara_clean:
                    nomor_perkara_clean = re.sub(r"[^\w]", "", url_detail.split("/")[-1])

                if nomor_perkara_clean in NOMOR_TERUNDUH:
                    print(f"[SKIP] Nomor Perkara '{nomor_perkara}' sudah diunduh sebelumnya.")
                    continue

                kandidat_pdf = cari_url_pdf(soup_detail)
                if not kandidat_pdf:
                    print("[INFO] Dokumen tidak memiliki berkas PDF.")
                    skipped_count += 1
                    continue

                pdf_bytes_ok = None
                for url_pdf in kandidat_pdf:
                    pdf_bytes_ok = fetch_pdf_bytes(driver, url_pdf)
                    if pdf_bytes_ok:
                        break

                if not pdf_bytes_ok:
                    print("[SKIP] File PDF gagal diunduh.")
                    skipped_count += 1
                    continue

                nama_pdf = nomor_perkara
                if nama_pdf:
                    nama_pdf = re.sub(r"[/\\]", "_", nama_pdf)
                    nama_pdf = re.sub(r"\s+", "_", nama_pdf)
                    nama_pdf = re.sub(r"_+", "_", nama_pdf).strip("_")
                if not nama_pdf or len(nama_pdf) < 3:
                    nama_pdf = f"kasasi_cbr_{downloaded_count + 1:03d}"

                path_pdf = os.path.join(PDF_DATA_DIR, f"{nama_pdf}.pdf")
                with open(path_pdf, "wb") as f:
                    f.write(pdf_bytes_ok)

                NOMOR_TERUNDUH.add(nomor_perkara_clean)
                with open(HISTORY_FILE, "a", encoding="utf-8") as f:
                    f.write(f"{nomor_perkara_clean}\n")

                downloaded_count += 1
                pbar.update(1)
                print(f"[SAVED] {path_pdf}")
                time.sleep(random.uniform(2.0, 3.0))

            page += 1

    except Exception as e:
        print(f"[ERROR]: {e}")
    finally:
        try:
            driver.quit()
        except Exception:
            pass
        pbar.close()

# Menjalankan pengambilan data khusus standar CBR
jalankan_scraper_pdf(limit_data=150)

[START] Mengaktifkan Browser Scraper Konsisten...


Progres Standarisasi Data CBR:   0%|          | 0/150 [00:00<?, ?it/s]


[PAGE 1] https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/mahkamah-agung/kategori/peradilan-anak-abh-1.html
[INFO] Terdeteksi 40 kandidat pada halaman ini.
[->] Membuka Dokumen: https://putusan3.mahkamahagung.go.id/direktori/putusan/11f162e21d1b7c72805d303933313131.html


Progres Standarisasi Data CBR:   1%|          | 1/150 [00:24<1:01:54, 24.93s/it]

[SAVED] data\pdf\4462_K_PID.SUS_2026.pdf
[->] Membuka Dokumen: https://putusan3.mahkamahagung.go.id/direktori/putusan/11f162e20071c59a88c2303933303233.html


...

Progres Standarisasi Data CBR:  99%|█████████▉| 149/150 [46:12<00:18, 18.83s/it]

[SAVED] data\pdf\9823_K_PID.SUS_2025.pdf
[->] Membuka Dokumen: https://putusan3.mahkamahagung.go.id/direktori/putusan/11f098e69708a9a4be52303833343139.html


Progres Standarisasi Data CBR: 100%|██████████| 150/150 [46:28<00:00, 18.02s/it]

[SAVED] data\pdf\8435_K_PID.SUS_2025.pdf


Progres Standarisasi Data CBR: 100%|██████████| 150/150 [46:33<00:00, 18.62s/it]



## 2 · [Poin a.ii–v] Konversi, Pembersihan, Validasi & Tokenisasi

Pipeline ini memproses seluruh PDF hasil scraper menjadi:
- Teks bersih paragraf utuh → `data/raw/case_NNN.txt`
- Token per baris + metadata header → `data/tokens/case_NNN.txt`
- Log pembersihan → `logs/cleaning.log`

| Tahap | Aksi |
|-------|------|
| **ii. Konversi** | PDF → plain text via `pdfplumber` dengan crop margin 15%/5% |
| **iii. Pembersihan** | Hapus header/footer/disclaimer/watermark, normalisasi spasi, lowercase |
| **iv. Validasi** | Periksa 9 marker konten putusan kasasi MA; minimal 5/9 harus ada |
| **v. Output** | Simpan `data/raw/*.txt`, `data/tokens/*.txt`, `logs/cleaning.log` |


## 1 · Import & Konfigurasi Path


In [1]:
import pdfplumber
import re
import os
import json
import pandas as pd
from datetime import datetime

from pathlib import Path

# ── Resolusi ROOT_DIR ───────────────────────────────────────────────
# Agar path data/ tetap valid baik notebook dijalankan dari root project
# maupun dari dalam folder notebook/ (working directory kernel kadang
# ikut lokasi file .ipynb). Cari ke atas folder yang punya subfolder data/;
# kalau belum ada (run pertama kali), fallback: naik 1 level jika cwd = "notebook".
def _tentukan_root_dir() -> Path:
    cwd = Path.cwd().resolve()
    p = cwd
    for _ in range(4):
        if (p / "data").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return cwd.parent if cwd.name == "notebook" else cwd

ROOT_DIR = _tentukan_root_dir()

PDF_DIR   = os.path.join(ROOT_DIR, 'data', 'pdf')
RAW_DIR   = os.path.join(ROOT_DIR, 'data', 'raw')
TOKEN_DIR = os.path.join(ROOT_DIR, 'data', 'tokens')
LOG_PATH  = os.path.join(ROOT_DIR, 'logs', 'cleaning.log')

os.makedirs(RAW_DIR,   exist_ok=True)
os.makedirs(TOKEN_DIR, exist_ok=True)
os.makedirs(os.path.join(ROOT_DIR, 'logs'), exist_ok=True)

pdf_files = sorted(f for f in os.listdir(PDF_DIR) if f.lower().endswith('.pdf'))
print(f'Direktori PDF  : {PDF_DIR}')
print(f'Direktori RAW  : {RAW_DIR}')
print(f'Direktori TOKEN: {TOKEN_DIR}')
print(f'Log            : {LOG_PATH}')
print(f'\nJumlah PDF ditemukan: {len(pdf_files)}')

Direktori PDF  : D:\Documents\SEMESTER6\Penalaran Komputer\Tugas 3(31)\data\pdf
Direktori RAW  : D:\Documents\SEMESTER6\Penalaran Komputer\Tugas 3(31)\data\raw
Direktori TOKEN: D:\Documents\SEMESTER6\Penalaran Komputer\Tugas 3(31)\data\tokens
Log            : D:\Documents\SEMESTER6\Penalaran Komputer\Tugas 3(31)\logs\cleaning.log

Jumlah PDF ditemukan: 150


### 2b · [Poin a.iii] Pola Pembersihan — Header / Footer / Watermark

Pola regex disusun dari yang paling spesifik ke umum agar tidak ada *over-removal*.
Setiap pola disubstitusi dengan spasi sehingga batas kata tetap terjaga.


In [2]:
# Pola 1: Disclaimer Mahkamah Agung (muncul di setiap halaman)
POLA_DISCLAIMER = re.compile(
    r'disclaimer\s+kepaniteraan.*?(?:halaman\s*\d+|$)',
    re.IGNORECASE | re.DOTALL
)

# Pola 2: Header berulang "halaman N dari N halaman putusan nomor ..."
POLA_HEADER_HALAMAN = re.compile(
    r'halaman\s+\d+\s+dari\s+\d+\s+halaman\s+putusan\s+nomor\s+[\w./\-]+',
    re.IGNORECASE
)

# Pola 3: Baris berisi hanya nomor halaman
POLA_NO_HALAMAN = re.compile(r'(?m)^\s*\d{1,3}\s*$')

# Pola 4: "putusan nomor ..." berdiri sendiri di baris
POLA_HEADER_PUTUSAN = re.compile(
    r'(?m)^putusan\s+nomor\s+\d+[\w./\\-]+\s*$',
    re.IGNORECASE
)

# Pola 5: Prefix noise PDF MA: 'gnomor' → 'nomor', 'gbahwa' → 'bahwa'
POLA_PREFIX_NOISE = re.compile(
    r'(?<!\w)g(?=nomor|bahwa|membaca|menimbang|mengingat|mengadili|putusan)',
    re.IGNORECASE
)

POLA_HAPUS = [POLA_DISCLAIMER, POLA_HEADER_HALAMAN, POLA_NO_HALAMAN,
              POLA_HEADER_PUTUSAN, POLA_PREFIX_NOISE]

print(f'Jumlah pola pembersihan: {len(POLA_HAPUS)}')
print('  1. Disclaimer kepaniteraan MA')
print('  2. Header halaman N dari N')
print('  3. Baris nomor halaman saja')
print('  4. Header "putusan nomor ..." standalone')
print('  5. Prefix noise konsonan PDF (g+kata_kunci)')

Jumlah pola pembersihan: 5
  1. Disclaimer kepaniteraan MA
  2. Header halaman N dari N
  3. Baris nomor halaman saja
  4. Header "putusan nomor ..." standalone
  5. Prefix noise konsonan PDF (g+kata_kunci)


### 2c · [Poin a.ii] Fungsi Ekstraksi Teks dari PDF

In [3]:
def ekstrak_pdf(path_pdf: str) -> tuple:
    """
    Ekstrak teks dari PDF putusan MA menggunakan pdfplumber.

    Crop margin digunakan untuk membuang:
    - Teks dekoratif di tepi kiri/kanan (watermark vertikal, nomor margin)
    - Header/footer di atas/bawah halaman
    - Karakter dengan koordinat y negatif (di luar area halaman)

    Returns
    -------
    tuple : (teks_gabungan: str, jumlah_halaman: int)
    """
    halaman_teks = []
    with pdfplumber.open(path_pdf) as pdf:
        n_pages = len(pdf.pages)
        for hal in pdf.pages:
            lebar  = hal.width
            tinggi = hal.height
            # bbox: (x0, top, x1, bottom) — koordinat dalam poin PDF
            area = hal.within_bbox((
                max(0, lebar  * 0.15),   # kiri: potong 15%
                max(0, tinggi * 0.05),   # atas: potong 5%
                lebar  * 0.85,           # kanan: potong 15%
                tinggi * 0.95,           # bawah: potong 5%
            ), relative=False)
            teks = area.extract_text(x_tolerance=3, y_tolerance=3)
            if teks:
                halaman_teks.append(teks)
    return '\n'.join(halaman_teks), n_pages


print('Fungsi ekstrak_pdf() siap.')
print('  Crop margin: kiri 15%, kanan 15%, atas 5%, bawah 5%')

Fungsi ekstrak_pdf() siap.
  Crop margin: kiri 15%, kanan 15%, atas 5%, bawah 5%


### 2d · [Poin a.iii] Fungsi Pembersihan Teks

In [4]:
def hapus_baris_noise(teks: str) -> str:
    """
    Hapus baris yang mayoritas berisi karakter tunggal terpisah spasi.
    Contoh noise PDF MA: 'a i s e n o d n i k i l b u p e' (teks dekoratif vertikal).
    Threshold: >70% token dengan panjang ≤2 karakter → buang baris.
    """
    baris_bersih = []
    for baris in teks.splitlines():
        token = baris.strip().split()
        if len(token) < 3:
            baris_bersih.append(baris)
            continue
        pendek = sum(1 for t in token if len(t) <= 2)
        if (pendek / len(token)) < 0.7:
            baris_bersih.append(baris)
    return '\n'.join(baris_bersih)


def bersihkan_teks(teks: str) -> str:
    """
    Pipeline pembersihan teks PDF putusan MA:
    1. Lowercase
    2. Hapus baris noise karakter vertikal (safety-net)
    3. Hapus footer/header/disclaimer via regex
    4. Pertahankan karakter yang relevan secara hukum
    5. Normalisasi whitespace

    Parameters
    ----------
    teks : str — teks mentah hasil ekstraksi PDF

    Returns
    -------
    str — teks bersih siap diproses tahap 02
    """
    # Langkah 1: lowercase lebih awal agar pola noise case-insensitive bekerja
    teks = teks.lower()

    # Langkah 2: hapus baris noise karakter vertikal
    teks = hapus_baris_noise(teks)

    # Langkah 3: hapus footer/header/disclaimer
    for pola in POLA_HAPUS:
        teks = pola.sub(' ', teks)

    # Langkah 4: pertahankan huruf, angka, spasi, newline, dan tanda baca hukum
    # Karakter '/' dipertahankan agar nomor perkara (123/K/Pid.Sus/2022) tetap utuh
    teks = re.sub(r"[^a-z0-9\s.,;:()'\-/\n]", ' ', teks)

    # Langkah 5: normalisasi whitespace
    teks = re.sub(r'[ \t]+',  ' ',   teks)
    teks = re.sub(r' *\n *',  '\n',  teks)
    teks = re.sub(r'\n{3,}', '\n\n', teks)
    teks = '\n'.join(line.strip() for line in teks.splitlines())

    return teks.strip()


print('Fungsi hapus_baris_noise() dan bersihkan_teks() siap.')

Fungsi hapus_baris_noise() dan bersihkan_teks() siap.


### 2e · [Poin a.iv] Marker Validasi Kelengkapan Dokumen

Validasi memastikan teks yang tersimpan merupakan putusan kasasi MA yang lengkap
(minimal 80% isi tersedia). Diukur dengan 9 penanda struktural khas putusan kasasi.


In [5]:
# Penanda struktural khas putusan kasasi Mahkamah Agung RI
MARKER = [
    'demi keadilan berdasarkan ketuhanan',   # kepala putusan
    'mahkamah agung',                         # pengadilan
    'memori kasasi',                          # argumen pemohon kasasi
    'menimbang',                              # pertimbangan hukum
    'terdakwa',                               # subjek perkara
    'penuntut umum',                          # pihak jaksa
    'mengadili',                              # amar putusan
    'membebankan biaya perkara',              # putusan biaya
    'judex facti',                            # istilah kasasi — merujuk pengadilan bawah
]

THRESHOLD_MARKER  = 5     # minimal 5/9 marker struktural harus ada
THRESHOLD_CHAR    = 500   # minimal 500 karakter setelah cleaning
THRESHOLD_RASIO   = 0.80  # minimal 80% isi putusan tersedia (char_bersih / char_mentah)

def hitung_marker(teks: str) -> int:
    """Hitung berapa penanda struktural yang ada dalam teks."""
    return sum(1 for m in MARKER if m in teks)

def validasi_dokumen(teks_bersih: str, teks_mentah: str = '') -> tuple:
    """
    Validasi kelengkapan dokumen putusan kasasi MA.

    Dua kriteria utama (keduanya harus terpenuhi):
    1. Rasio karakter: char_bersih / char_mentah >= 80%
       Memastikan proses cleaning tidak membuang terlalu banyak konten.
    2. Marker struktural: minimal 5/9 penanda khas putusan kasasi harus ada.
       Memastikan dokumen adalah putusan kasasi MA yang lengkap.

    Parameters
    ----------
    teks_bersih : str — teks setelah cleaning
    teks_mentah : str — teks mentah sebelum cleaning (untuk hitung rasio)

    Returns
    -------
    tuple : (lulus: bool, n_marker: int, rasio_char: float, keterangan: str)
    """
    n_char   = len(teks_bersih)
    n_marker = hitung_marker(teks_bersih)

    # Hitung rasio kelengkapan karakter
    n_mentah   = len(teks_mentah) if teks_mentah else n_char
    rasio_char = (n_char / n_mentah) if n_mentah > 0 else 1.0

    # Kriteria 1: teks terlalu pendek (kemungkinan gagal ekstraksi)
    if n_char < THRESHOLD_CHAR:
        ket = f'TERLALU_PENDEK — {n_char} karakter (min {THRESHOLD_CHAR})'
        return False, n_marker, rasio_char, ket

    # Kriteria 2: rasio karakter < 80% (terlalu banyak konten hilang saat cleaning)
    if rasio_char < THRESHOLD_RASIO:
        ket = f'RASIO_RENDAH — {rasio_char:.1%} karakter tersisa (min {THRESHOLD_RASIO:.0%})'
        return False, n_marker, rasio_char, ket

    # Kriteria 3: marker struktural kurang (bukan putusan kasasi lengkap)
    if n_marker < THRESHOLD_MARKER:
        ket = f'KURANG_MARKER — {n_marker}/{len(MARKER)} (min {THRESHOLD_MARKER})'
        return False, n_marker, rasio_char, ket

    ket = f'LULUS — {n_char:,} karakter, rasio {rasio_char:.1%}, {n_marker}/{len(MARKER)} marker'
    return True, n_marker, rasio_char, ket

print(f'Jumlah marker    : {len(MARKER)}')
print(f'Threshold marker : min {THRESHOLD_MARKER}/{len(MARKER)}')
print(f'Threshold char   : min {THRESHOLD_CHAR} karakter')
print(f'Threshold rasio  : min {THRESHOLD_RASIO:.0%} isi putusan tersedia (char_bersih/char_mentah)')

Jumlah marker    : 9
Threshold marker : min 5/9
Threshold char   : min 500 karakter
Threshold rasio  : min 80% isi putusan tersedia (char_bersih/char_mentah)


### 2f · [Poin a.v] Fungsi Tokenisasi

In [6]:
# Format nomor perkara MA: '123 K/Pid.Sus-Anak/2022' atau '456/Pid.Sus/2021'
POLA_NOMOR_PERKARA = re.compile(
    r'\d+\s*/\s*(?:k\s*/\s*)?pid[\w.\-]*/\s*20\d{2}',
    re.IGNORECASE
)
POLA_TAHUN    = re.compile(r'^(19|20)\d{2}$')
KONTEKS_ANGKA = {'pasal', 'nomor', 'ayat', 'jo', 'uu', 'tahun', 'ps'}


def tokenisasi(teks_bersih: str) -> list:
    """
    Tokenisasi teks bersih putusan kasasi MA.

    Perlakuan khusus:
    - Nomor perkara MA diangkat sebagai token utuh (tidak dipecah per karakter)
    - Token 1 karakter dibuang (noise)
    - Angka disimpan jika merupakan tahun atau didahului kata konteks hukum

    Parameters
    ----------
    teks_bersih : str — output dari bersihkan_teks()

    Returns
    -------
    list[str] — daftar token
    """
    tokens = []
    for baris in teks_bersih.splitlines():
        baris = baris.strip()
        if not baris:
            continue

        daftar_np   = POLA_NOMOR_PERKARA.findall(baris)
        baris_kerja = POLA_NOMOR_PERKARA.sub('XNPX', baris)
        kata_list   = re.findall(r'[a-z0-9]+', baris_kerja)

        token_sebelum = ''
        idx_np        = 0

        for t in kata_list:
            if t == 'xnpx':
                if idx_np < len(daftar_np):
                    tokens.append(daftar_np[idx_np].lower())
                    idx_np += 1
                token_sebelum = 'xnpx'
                continue

            if len(t) < 2:
                token_sebelum = t
                continue

            if t.isdigit():
                if POLA_TAHUN.match(t) or token_sebelum in KONTEKS_ANGKA:
                    tokens.append(t)
                token_sebelum = t
                continue

            tokens.append(t)
            token_sebelum = t

    return tokens


print('Fungsi tokenisasi() siap.')

Fungsi tokenisasi() siap.


### 2g · [Poin a.v] Pipeline Utama — Proses Semua PDF

Setiap PDF diproses secara berurutan:
1. Ekstrak teks dari PDF (pdfplumber + crop margin)
2. Bersihkan teks (hapus noise, normalisasi)
3. Validasi kelengkapan (marker)
4. Tokenisasi
5. Simpan `data/raw/case_NNN.txt` — teks bersih paragraf utuh
6. Simpan `data/tokens/case_NNN.txt` — token + header metadata
7. Catat ke `logs/cleaning.log`


In [7]:
pdf_files = sorted(f for f in os.listdir(PDF_DIR) if f.lower().endswith('.pdf'))
print(f'Memproses {len(pdf_files)} file PDF ...\n')

log_rows = []
gagal    = []

for i, nama_pdf in enumerate(pdf_files, 1):
    path_pdf = os.path.join(PDF_DIR, nama_pdf)
    nama_base = f'case_{i:03d}'
    nama_txt  = f'{nama_base}.txt'
    path_txt  = os.path.join(RAW_DIR, nama_txt)
    waktu_proses = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # Nilai default — dipakai jika error terjadi sebelum variabel terdefinisi
    rasio_char  = 0.0
    n_marker    = 0
    n_pages     = 0
    char_mentah = 0
    char_bersih = 0
    jumlah_token = 0
    keterangan  = ''
    status      = 'ERROR'

    try:
        # Tahap a.ii: Konversi PDF ke teks
        teks_mentah, n_pages = ekstrak_pdf(path_pdf)
        char_mentah          = len(teks_mentah)

        # Tahap a.iii: Pembersihan
        teks_bersih = bersihkan_teks(teks_mentah)
        char_bersih = len(teks_bersih)

        # Tahap a.iv: Validasi (rasio 80% + marker)
        # teks_mentah dikirim agar rasio char_bersih/char_mentah dapat dihitung
        lulus, n_marker, rasio_char, keterangan = validasi_dokumen(teks_bersih, teks_mentah)
        status = 'TERSIMPAN_OK' if lulus else 'PERLU_CEK'

        # Tokenisasi
        tokens       = tokenisasi(teks_bersih)
        jumlah_token = len(tokens)

        # Tahap a.v: Simpan teks bersih ke data/raw/
        with open(path_txt, 'w', encoding='utf-8') as f:
            f.write(teks_bersih)

        # Tahap a.v: Simpan token .txt
        path_token_txt = os.path.join(TOKEN_DIR, nama_txt)
        header = (
            f'# CASE_ID    : {nama_base}\n'
            f'# FILENAME   : {nama_pdf}\n'
            f'# JENIS      : putusan kasasi MA\n'
            f'# N_PAGES    : {n_pages}\n'
            f'# N_TOKENS   : {jumlah_token}\n'
            f'# STATUS     : {status}\n'
            f'# CLEANED_ON : {waktu_proses}\n'
            f'# {"="*60}\n'
        )
        with open(path_token_txt, 'w', encoding='utf-8') as f:
            f.write(header)
            f.write('\n'.join(tokens))

        # Tahap a.v: Simpan token .json
        path_token_json = os.path.join(TOKEN_DIR, f'{nama_base}.json')
        token_json_obj = {
            'metadata': {
                'case_id'    : nama_base,
                'filename'   : nama_pdf,
                'jenis'      : 'putusan kasasi MA',
                'n_pages'    : n_pages,
                'n_tokens'   : jumlah_token,
                'status'     : status,
                'cleaned_on' : waktu_proses,
            },
            'tokens': tokens
        }
        with open(path_token_json, 'w', encoding='utf-8') as f:
            json.dump(token_json_obj, f, ensure_ascii=False, indent=2)

        # Cek nomor perkara
        np_found = POLA_NOMOR_PERKARA.findall(teks_bersih)
        np_info  = np_found[0] if np_found else '(tidak terdeteksi)'

        print(f'[{"OK" if lulus else "!?"}] {nama_base} <- {nama_pdf[:50]}')
        print(f'     Halaman : {n_pages} | Karakter bersih: {char_bersih:,} | Token: {jumlah_token:,}')
        print(f'     Marker  : {n_marker}/{len(MARKER)} | Rasio: {rasio_char:.1%} | Nomor: {np_info}')
        print(f'     Status  : {keterangan}')
        print(f'     Saved   : {nama_txt} + {nama_base}.json')

    except Exception as e:
        status     = 'ERROR'
        keterangan = str(e)
        # rasio_char dan variabel lain sudah diberi nilai default 0/0.0 di atas blok try
        # sehingga log_rows tidak akan melempar NameError
        gagal.append(nama_pdf)
        print(f'[ERR] {nama_pdf} -> {e}')

    log_rows.append({
        'timestamp'    : waktu_proses,
        'file_pdf'     : nama_pdf,
        'file_output'  : nama_txt,
        'status'       : status,
        'keterangan'   : keterangan,
        'n_pages'      : n_pages,
        'char_mentah'  : char_mentah,
        'char_bersih'  : char_bersih,
        'rasio_char'   : round(rasio_char, 4),
        'jumlah_token' : jumlah_token,
    })

# Simpan cleaning.log
df_log = pd.DataFrame(log_rows)
df_log.to_csv(LOG_PATH, index=False, encoding='utf-8-sig')

n_ok   = (df_log['status'] == 'TERSIMPAN_OK').sum()
n_cek  = (df_log['status'] == 'PERLU_CEK').sum()
n_err  = (df_log['status'] == 'ERROR').sum()

print(f'\n{"="*60}')
print(f'PREPROCESSING SELESAI')
print(f'  Total PDF diproses : {len(pdf_files)}')
print(f'  TERSIMPAN_OK       : {n_ok}')
print(f'  PERLU_CEK          : {n_cek}')
print(f'  ERROR              : {n_err}')
print(f'  Gagal              : {gagal if gagal else "tidak ada"}')
print(f'\nOutput per dokumen:')
print(f'  data/raw/case_NNN.txt         <- teks bersih paragraf utuh')
print(f'  data/tokens/case_NNN.txt      <- token per baris + header #')
print(f'  data/tokens/case_NNN.json     <- metadata + tokens')
print(f'  logs/cleaning.log             <- CSV riwayat preprocessing')

Memproses 150 file PDF ...

[OK] case_001 <- 10004_K_PID.SUS_2025.pdf
     Halaman : 7 | Karakter bersih: 11,834 | Token: 1,546
     Marker  : 9/9 | Rasio: 99.4% | Nomor: (tidak terdeteksi)
     Status  : LULUS — 11,834 karakter, rasio 99.4%, 9/9 marker
     Saved   : case_001.txt + case_001.json
[OK] case_002 <- 10051_K_PID.SUS_2025.pdf
     Halaman : 8 | Karakter bersih: 14,323 | Token: 1,858
     Marker  : 8/9 | Rasio: 99.5% | Nomor: 10/pid.sus-anak/2025
     Status  : LULUS — 14,323 karakter, rasio 99.5%, 8/9 marker
     Saved   : case_002.txt + case_002.json
[OK] case_003 <- 10053_K_PID.SUS_2025.pdf
     Halaman : 10 | Karakter bersih: 17,093 | Token: 2,226
     Marker  : 7/9 | Rasio: 96.7% | Nomor: 4/pid.sus-anak/2025
     Status  : LULUS — 17,093 karakter, rasio 96.7%, 7/9 marker
     Saved   : case_003.txt + case_003.json
[OK] case_004 <- 10054_K_PID.SUS_2025.pdf
     Halaman : 9 | Karakter bersih: 14,925 | Token: 1,969
     Marker  : 7/9 | Rasio: 97.0% | Nomor: (tidak terdetek

## 3 · [Poin a.iv] Verifikasi Hasil & Cleaning Log

Preview teks bersih dan token dari salah satu dokumen, serta tampilan tabel cleaning.log.


In [8]:
txt_files = sorted(f for f in os.listdir(RAW_DIR) if f.endswith('.txt'))
print(f'File txt di data/raw/: {len(txt_files)}')

if txt_files:
    nama_sample = txt_files[0]
    nama_base   = nama_sample.replace('.txt', '')

    # ── Preview teks bersih (data/raw/) ─────────────────────────────
    path_raw = os.path.join(RAW_DIR, nama_sample)
    with open(path_raw, encoding='utf-8') as f:
        isi_raw = f.read()
    print(f'\n=== Preview RAW: {nama_sample} (600 karakter pertama) ===')
    print(isi_raw[:600])
    print(f'  ... (total {len(isi_raw):,} karakter)')

    # ── Preview token .txt (data/tokens/) ────────────────────────────
    path_tok = os.path.join(TOKEN_DIR, nama_sample)
    with open(path_tok, encoding='utf-8') as f:
        baris_semua = f.read().splitlines()
    baris_header = [b for b in baris_semua if b.startswith('#')]
    baris_token  = [b for b in baris_semua if not b.startswith('#') and b.strip()]

    print(f'\n=== Preview TOKENS (.txt): {nama_sample} ===')
    print('--- Header metadata ---')
    for h in baris_header:
        print(h)
    print('\n--- 30 token pertama ---')
    print(' | '.join(baris_token[:30]))
    print(f'  ... (total {len(baris_token):,} token)')

    # ── Preview token .json (data/tokens/) ───────────────────────────
    path_json = os.path.join(TOKEN_DIR, f'{nama_base}.json')
    with open(path_json, encoding='utf-8') as f:
        token_obj = json.load(f)

    print(f'\n=== Preview TOKENS (.json): {nama_base}.json ===')
    print('--- metadata ---')
    for k, v in token_obj['metadata'].items():
        print(f'  {k:12s}: {v}')
    print(f'\n--- tokens (30 pertama) ---')
    print(token_obj['tokens'][:30])
    print(f'  ... (total {len(token_obj["tokens"]):,} token)')

    # Verifikasi nomor perkara tersimpan utuh di token
    np_tokens = [t for t in token_obj['tokens'] if '/' in t and 'k' in t]
    if np_tokens:
        print(f'\nToken nomor perkara   : {np_tokens[:3]}')

# ── Tabel cleaning.log ───────────────────────────────────────────────
print('\n=== CLEANING LOG (10 baris pertama) ===')
df_log = pd.read_csv(LOG_PATH, encoding='utf-8-sig')
display(df_log[['file_pdf', 'file_output', 'status', 'n_pages',
                'char_bersih', 'rasio_char', 'jumlah_token', 'keterangan']].head(10))

print(f'\nSUMARY CLEANING LOG:')
print(df_log['status'].value_counts().to_string())
print(f'\nTotal dokumen valid (TERSIMPAN_OK + PERLU_CEK): ',
      df_log['status'].isin(['TERSIMPAN_OK','PERLU_CEK']).sum())
print('\nLanjut ke : 02_case_representation.ipynb')

File txt di data/raw/: 150

=== Preview RAW: case_001.txt (600 karakter pertama) ===
putusan.mahkamahagung.go.id
r
g
nomor 10004 k/pid.sus/2025
n
demi keadilan berdasarkan ketuhanan yang maha esa
megmeriksa perkara pidana khusus pada tingkat kasasi yang dimohdonkan oleh
penuntut umum pada kejaksaan negeri kotawaringin barat,n telah memutus
a
perkara anak:
i
nama : muhammad rendi pang alila alias tole
h k
bin susanto;
i
tempat lahir : kotawaringin barat; l
b
umur/tanggal lahir : 14 tahun /20 desember 2010;
jenis kelamin : laki-laki; u
kewarganegaraan : indonespia;
tempat tinggal : desa kumpai batu bawah rt 010 rw 003,
e
kecamatan arut selatan, kabupaten kotawaringin
r
barat, pr
  ... (total 11,834 karakter)

=== Preview TOKENS (.txt): case_001.txt ===
--- Header metadata ---
# CASE_ID    : case_001
# FILENAME   : 10004_K_PID.SUS_2025.pdf
# JENIS      : putusan kasasi MA
# N_PAGES    : 7
# N_TOKENS   : 1546
# STATUS     : TERSIMPAN_OK
# CLEANED_ON : 2026-06-26 20:46:12
# ================

,file_pdf,file_output,status,n_pages,char_bersih,rasio_char,jumlah_token,keterangan
0,10004_K_PID.SUS_2025.pdf,case_001.txt,TERSIMPAN_OK,7,11834,0.9942,1546,"LULUS — 11,834 karakter, rasio 99.4%, 9/9 marker"
1,10051_K_PID.SUS_2025.pdf,case_002.txt,TERSIMPAN_OK,8,14323,0.9953,1858,"LULUS — 14,323 karakter, rasio 99.5%, 8/9 marker"
2,10053_K_PID.SUS_2025.pdf,case_003.txt,TERSIMPAN_OK,10,17093,0.9672,2226,"LULUS — 17,093 karakter, rasio 96.7%, 7/9 marker"
3,10054_K_PID.SUS_2025.pdf,case_004.txt,TERSIMPAN_OK,9,14925,0.9695,1969,"LULUS — 14,925 karakter, rasio 97.0%, 7/9 marker"
4,10063_K_PID.SUS_2025.pdf,case_005.txt,TERSIMPAN_OK,9,15172,0.9663,1988,"LULUS — 15,172 karakter, rasio 96.6%, 7/9 marker"
5,10115_K_PID.SUS_2025.pdf,case_006.txt,TERSIMPAN_OK,7,12422,0.9699,1644,"LULUS — 12,422 karakter, rasio 97.0%, 8/9 marker"
6,10169_K_PID.SUS_2025.pdf,case_007.txt,TERSIMPAN_OK,9,15386,0.9701,2031,"LULUS — 15,386 karakter, rasio 97.0%, 8/9 marker"
7,10222_K_PID.SUS_2025.pdf,case_008.txt,TERSIMPAN_OK,7,11579,0.9682,1528,"LULUS — 11,579 karakter, rasio 96.8%, 8/9 marker"
8,10296_K_PID.SUS_2025.pdf,case_009.txt,TERSIMPAN_OK,8,14566,0.9715,1918,"LULUS — 14,566 karakter, rasio 97.2%, 7/9 marker"
9,10411_K_PID.SUS_2025.pdf,case_010.txt,TERSIMPAN_OK,7,11813,0.9668,1553,"LULUS — 11,813 karakter, rasio 96.7%, 8/9 marker"



SUMARY CLEANING LOG:
status
TERSIMPAN_OK    149
PERLU_CEK         1

Total dokumen valid (TERSIMPAN_OK + PERLU_CEK):  150

Lanjut ke : 02_case_representation.ipynb
